# GameGuideLM — Stardew Valley Release Demo

This notebook reproduces the offline course-release demo. It does **not** download model weights and does not claim Qwen/QLoRA quality gains or speculative-decoding speedup.

Verified release scope:

- 505 structured Stardew records
- 317 acquisition relations
- 25 offline guide pages and 100 searchable chunks
- 100 bilingual deterministic regression candidates
- 159 grounded training records and 17 validation records
- 1,262 audited legacy SFT candidates
- 171 passing repository tests

**Truth boundary:** the 100-case result is an engineering regression against tracked rules, not an independently human-approved factual benchmark. Independent source review and GPU experiments remain pending.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

CANDIDATES = [
    Path('/content/llm_project'),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = next(
    (path.resolve() for path in CANDIDATES if (path / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the project. In Colab, clone or unzip it to /content/llm_project.'
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')

# The offline demo can run directly from the source tree. Set this to True in a
# fresh Colab runtime when dependencies still need to be installed.
INSTALL_EDITABLE_PACKAGE = False
if INSTALL_EDITABLE_PACKAGE:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-e', '.'],
        check=True,
    )
else:
    print('Using the source tree directly; editable installation skipped.')


## 1. Rebuild the deterministic release

`--skip-tests` keeps the live notebook fast. Remove it for the full 171-test release build.

In [ ]:
RUN_FULL_TEST_SUITE = False

command = [sys.executable, 'scripts/build_stardew_release.py']
if not RUN_FULL_TEST_SUITE:
    command.append('--skip-tests')

subprocess.run(command, check=True)


## 2. Inspect the release manifests

In [ ]:
import json
from IPython.display import display
import pandas as pd

release = json.loads((PROJECT_ROOT / 'RELEASE_VALIDATION.json').read_text(encoding='utf-8'))
catalog = json.loads(
    (PROJECT_ROOT / 'data/stardew/catalog/snapshot_manifest.json').read_text(encoding='utf-8')
)
evaluation = json.loads(
    (PROJECT_ROOT / 'data/stardew/evaluation/manifest_v1.json').read_text(encoding='utf-8')
)

summary = pd.DataFrame([
    {'Metric': 'Repository tests', 'Value': release['python_test_suite']['passed']},
    {'Metric': 'Structured records', 'Value': release['stardew']['record_count']},
    {'Metric': 'Acquisition relations', 'Value': release['stardew']['acquisition_relations']},
    {'Metric': 'Guide pages', 'Value': release['stardew']['guide_pages']},
    {'Metric': 'Guide chunks', 'Value': release['stardew']['guide_chunks']},
    {'Metric': 'Regression cases passed', 'Value': f"{release['stardew']['regression_passed']}/{release['stardew']['regression_examples']}"},
    {'Metric': 'Human review status', 'Value': release['stardew']['human_review_status']},
])
display(summary)

record_counts = catalog.get('record_type_counts', catalog.get('counts_by_type', {}))
display(pd.DataFrame(sorted(record_counts.items()), columns=['Record type', 'Count']))

print('Evaluation language distribution:', evaluation.get('language_distribution'))
print('Evaluation status distribution:', evaluation.get('status_distribution'))


## 3. Run four deterministic assistant examples

These commands exercise four distinct safety behaviors: player-state calculation, bilingual alias lookup, partial coverage, and safe refusal.

In [ ]:
def run_demo(*args: str) -> None:
    command = [sys.executable, 'scripts/chat_gameguide.py', *args]
    print('$', ' '.join(command))
    result = subprocess.run(command, text=True, capture_output=True, check=True)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    print('-' * 100)

run_demo(
    '--game', 'stardew', '--season', 'fall', '--day', '15',
    '秋季第15天种Pumpkin还能收获吗？',
)
run_demo('--game', 'stardew', '阿比盖尔喜欢什么礼物？')
run_demo('--game', 'stardew', 'What is in the remixed River Fish Bundle?')
run_demo('--game', 'stardew', 'Where can I get Dragon Tractor?')


## 4. Open the self-contained showcase

In [ ]:
from IPython.display import HTML, display

showcase_path = PROJECT_ROOT / 'demo/stardew_showcase.html'
if not showcase_path.exists():
    raise FileNotFoundError(showcase_path)

display(HTML(showcase_path.read_text(encoding='utf-8')))


## 5. Optional full validation

Enable this cell before final submission. It can take longer than the demo path.

In [ ]:
RUN_FINAL_VALIDATION = False

if RUN_FINAL_VALIDATION:
    subprocess.run([sys.executable, 'scripts/validate_stardew_release.py'], check=True)
    subprocess.run([sys.executable, 'scripts/validate_release.py', '--skip-pytest'], check=True)
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
else:
    print('Skipped. Set RUN_FINAL_VALIDATION=True for the complete submission check.')


## 6. Optional GPU experiment — deliberately disabled

The intended measured comparison is:

1. Qwen3-4B target-only
2. Qwen3-0.6B draft → Qwen3-4B target
3. TinyQwenDraft → Qwen3-4B target

Only enable a GPU experiment after model checkpoints are available and the benchmark protocol records GPU type, precision, warm-up, prompt/output lengths, draft length, acceptance rate, TTFT, TPOT, throughput, memory, and exact greedy-output equality. Do not report estimated performance values.

In [ ]:
RUN_GPU_EXPERIMENTS = False

if RUN_GPU_EXPERIMENTS:
    raise NotImplementedError(
        'Provide validated local model/checkpoint paths and run the repository benchmark scripts '
        'under a controlled warm-GPU protocol before enabling this cell.'
    )
else:
    print('GPU experiments remain pending; no performance claim is made by this notebook.')
